In [ ]:
import numpy as np
import pandas as pd
import sys, os
import csv
import json
import matplotlib.pyplot as plt

bonsai_path = '/scicore/home/nimwegen/degroo0000/Bonsai-data-representation'
sys.path.append(bonsai_path)
from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import get_pdists_on_tree

In [ ]:
bonsai_results_folder = '/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets/C_elegans_timecourse/Bonsai/final_bonsai_zscore1.0_tmpStartnon_refined_tree'

In [ ]:
with open(os.path.join(bonsai_results_folder, 'metadata.json'), 'r') as file:
    metadata = json.load(file)
cell_ids = metadata['cellIds']

In [ ]:
annotation_path = '/scicore/home/nimwegen/degroo0000/sc_datasets/C_elegans_packer/Waterston/annotation/orig_annot.tsv'
annotation = pd.read_csv(annotation_path, index_col=0, sep='\t')

In [ ]:
time_annot = 'time_point'
time_annot = 'embryo_time_bin'

In [ ]:
time_points, counts = list(np.unique(annotation[time_annot], return_counts=True))

In [ ]:
list(zip(time_points, map(int, counts)))

In [ ]:
cell_ids_per_tp = {tp: list(annotation.index[annotation[time_annot] == tp]) for tp in time_points}
cell_id_to_idx = {cid: i for i, cid in enumerate(cell_ids)}
indices_per_tp = {}
pdists_per_tp = {}
for tp in time_points:
    cell_ids_subset = cell_ids_per_tp[tp]
    print(len(cell_ids_subset))
    indices_per_tp[tp] = np.array([cell_id_to_idx[cell_id] for cell_id in cell_ids_per_tp[tp]])

In [ ]:
cell_ids_per_tp = {tp: list(annotation.index[annotation[time_annot] == tp]) for tp in time_points}
cell_id_to_idx = {cid: i for i, cid in enumerate(cell_ids)}
indices_per_tp = {}
pdists_per_tp = {}
for tp in time_points:
    cell_ids_subset = cell_ids_per_tp[tp]
    print(len(cell_ids_subset))
    indices_per_tp[tp] = np.array([cell_id_to_idx[cell_id] for cell_id in cell_ids_per_tp[tp]])
    nwk_file = os.path.join(bonsai_results_folder, 'tree.nwk')
    if not os.path.exists('pdists_{}.npy'.format(tp)):
        bonsai_pdists = get_pdists_on_tree(nwk_file, cell_ids_subset)
        np.save('pdists_{}.npy'.format(tp), bonsai_pdists)
        print("Done with {}".format(tp))
    else:
        print("Skipping calculating pairwise distances on {} because results already stored.".format(tp))

In [ ]:
bins = np.linspace(0, 2.3, 200)  # choose once

fig, ax = plt.subplots()

for tp in time_points:
    if not os.path.exists('pdists_{}.npy'.format(tp)):
        continue
    print(tp)
    pairwise_dists = np.load('pdists_{}.npy'.format(tp))
    if len(pairwise_dists) < (300**2/2):
        continue
    counts, edges = np.histogram(pairwise_dists, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, counts, label=tp)

# ax.set_yscale('log')
ax.legend()
ax.set_xlabel("Pairwise distances between all cells of that timepoint")
ax.set_ylabel("Density")

# Try a different approach where you just calculate distances from the root

In [ ]:
from bonsai_scout.my_tree_layout import Layout_Tree
from scipy.sparse.csgraph import shortest_path
from scipy.sparse import csr_matrix


def get_pdists_from_root(nwk_file, cell_ids):
    tree = Layout_Tree()

    with open(nwk_file, "r") as f:
        nwk_str = f.readline()

    tree.from_newick(nwk_str=nwk_str)

    # Renumber vert_inds on tree such that they are in line with a depth-first search
    vertIndToNode, tree.nNodes = tree.root.renumber_verts(vertIndToNode={}, vert_count=0)
    tree.vert_ind_to_node = vertIndToNode
    tree.root.storeParent()

    root_id = tree.root.nodeId

    node_id_to_vert_ind = {node.nodeId: vert_ind for vert_ind, node in tree.vert_ind_to_node.items()}
    # Get indices between which distances are calculated
    root_ind = [node_id_to_vert_ind[root_id]]
    cell_inds = [node_id_to_vert_ind[node_id] for node_id in cell_ids]

    
    # edge_dict has keys source, target, dist, source_ind, target_ind
    edge_dict = tree.get_edge_dict(nodeIdToVertInd=node_id_to_vert_ind)

    cols = np.array(edge_dict['source_ind'])
    rows = np.array(edge_dict['target_ind'])
    weights = np.array(edge_dict['dist'])

    colsComplete = np.concatenate((cols, rows))
    rowsComplete = np.concatenate((rows, cols))
    weightsComplete = np.concatenate((weights, weights))
    nVerts = np.max(colsComplete) + 1
    distance_csr = csr_matrix((weightsComplete, (rowsComplete, colsComplete)), shape=(nVerts, nVerts))

    print("Done preparing inputs. Starting shortest-path algorithm from Scipy.")
    distances = shortest_path(distance_csr, method='auto', directed=False, return_predecessors=False,
                                         unweighted=False, overwrite=False, indices=root_ind)[:, cell_inds].flatten()
    return distances


In [ ]:
nwk_file = os.path.join(bonsai_results_folder, 'tree.nwk')
dists_to_root = get_pdists_from_root(nwk_file, cell_ids)

In [ ]:
dists_to_root.shape

In [ ]:
fig, ax = plt.subplots()
for tp in time_points:
    cell_inds_subset = indices_per_tp[tp]
    if len(cell_inds_subset) < 300:
        continue
    dists_to_root_subset = dists_to_root[cell_inds_subset]
    ax.hist(dists_to_root_subset, bins=300, label=tp, histtype='step', density=True, linewidth=2)
ax.legend()
ax.set_ylabel("Distance to root")

In [ ]:
len(cell_ids)

In [ ]:
[(tp, len(inds)) for tp, inds in indices_per_tp.items()]

In [ ]:
fig, ax = plt.subplots()

data = []
labels = []
colors = plt.cm.tab20(range(len(time_points)))
cat_to_col = {}
for ind_tp, tp in enumerate(time_points):
    cat_to_col[tp] = colors[ind_tp, :]
    if len(indices_per_tp[tp]) < 300:
        continue
    data.append(dists_to_root[indices_per_tp[tp]])
    labels.append(tp)
# data = [dists_to_root[indices_per_tp[tp]] for tp in time_points if len(indices_per_tp[tp]) > 300]

bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)

for box, label in zip(bp["boxes"], labels):
    box.set_facecolor(cat_to_col[label])

for element in ["whiskers", "caps", "medians"]:
    plt.setp(bp[element], color="black")

ax.set_xlabel("Timepoint")
ax.set_ylabel("Distance to root")

ax.tick_params(axis="x", labelrotation=45)

plt.savefig('boxplots_distance_to_root.svg')
plt.savefig('boxplots_distance_to_root.png', dpi=300)

In [ ]:
fig, ax = plt.subplots()

data = []
labels = []
for tp in time_points:
    if len(indices_per_tp[tp]) < 300:
        continue
    data.append(dists_to_root[indices_per_tp[tp]])
    labels.append(tp)

vp = ax.violinplot(
    data,
    showmedians=True,
    showmeans=False,
    showextrema=False
)

# Set x-axis labels
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)

# Color each violin
colors = plt.cm.tab20(range(len(labels)))
for body, color in zip(vp["bodies"], colors):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.8)

ax.boxplot(
    data,
    widths=0.15,
    showfliers=False,
    patch_artist=True,
    boxprops=dict(facecolor="white", edgecolor="black")
)

ax.set_xlabel("Timepoint")
ax.set_ylabel("Distance to root")